# Real-time Face Detection & Recognition - InsightFace + ArcFace embeddings

**Komputerisasi Industri Pertanian**

- **Nama:** Junervin  
- **NIM:** ................................  
- **Kelas:** ................................  
- **Dataset:** Facebank_TIP_2026
- **Metode:**
  - Face detection + landmark + alignment + embedding menggunakan `InsightFace FaceAnalysis`.
  - Recognition menggunakan ArcFace normalized embedding dan pencocokan cosine similarity ke facebank.

## 1. Setup library

Disarankan memakai **Runtime → Change runtime type → GPU** di Google Colab.  
Jika GPU tidak tersedia, notebook tetap dapat berjalan di CPU, tetapi FPS live camera akan lebih rendah.

In [ ]:
# Install package utama untuk Google Colab.

!pip -q install --upgrade "numpy>=1.26,<2.1" "pandas==2.2.2"

!pip -q install --upgrade \
  "insightface==0.7.3" \
  "onnxruntime-gpu>=1.17,<1.23" \
  "opencv-python-headless>=4.10,<4.13" \
  "tqdm" "scikit-learn" "matplotlib"

print("Instalasi selesai. Jika muncul pesan restart runtime, restart lalu lanjutkan dari cell berikutnya.")


## 2. Import, konfigurasi folder, dan mount Google Drive

Struktur dataset yang disarankan:

```text
PROJECT_DIR/
├── dataset/
│   ├── nama_orang_1/
│   │   ├── 001.jpg
│   │   ├── 002.jpg
│   ├── nama_orang_2/
│   │   ├── 001.jpg
│   │   ├── 002.jpg
├── facebank_insightface_buffalo_l.npz
└── logs/
```

Minimal praktis: **5–20 foto per orang**, variasikan arah wajah, pencahayaan, dan ekspresi. Untuk enrollment cepat, notebook ini juga menyediakan fungsi capture foto dari kamera.

In [ ]:
import os
import sys
import json
import time
import glob
import base64
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# KONFIGURASI UTAMA
# ------------------------------------------------------------
USE_DRIVE = True  # ubah ke False jika ingin menyimpan hanya di runtime Colab sementara

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab_Face_Recognition_InsightFace")
else:
    PROJECT_DIR = Path("/content/Colab_Face_Recognition_InsightFace")

DATASET_ROOT = PROJECT_DIR / "dataset"
LOG_DIR = PROJECT_DIR / "logs"

MODEL_PACK = "buffalo_l"       # default auto-download; akurat
DET_SIZE = (320, 320)          # turunkan ke (320, 320) jika live camera lambat
SIM_THRESHOLD = 0.42           # naikkan agar lebih ketat; turunkan jika terlalu banyak UNKNOWN
FACEBANK_PATH = PROJECT_DIR / f"facebank_insightface_{MODEL_PACK}.npz"

for p in [PROJECT_DIR, DATASET_ROOT, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("DATASET_ROOT:", DATASET_ROOT)
print("FACEBANK    :", FACEBANK_PATH)

## 3. Load model InsightFace

`FaceAnalysis` akan mengunduh model pack saat pertama kali dijalankan. Untuk Colab, model `buffalo_l` biasanya menjadi pilihan seimbang karena sudah mencakup detection, landmark/alignment, gender/age, dan recognition embedding.

In [ ]:
import onnxruntime as ort
from insightface.app import FaceAnalysis

def load_insightface_app(model_pack=MODEL_PACK, det_size=DET_SIZE):
    available = ort.get_available_providers()
    print("ONNXRuntime providers tersedia:", available)

    prefer_cuda = "CUDAExecutionProvider" in available
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if prefer_cuda else ["CPUExecutionProvider"]
    ctx_id = 0 if prefer_cuda else -1

    try:
        app = FaceAnalysis(name=model_pack, providers=providers)
        app.prepare(ctx_id=ctx_id, det_size=det_size)
        print(f"Model aktif: {model_pack} | provider: {providers[0]} | det_size: {det_size}")
        return app
    except Exception as e:
        print("Load GPU gagal, fallback ke CPU. Detail:", repr(e))
        app = FaceAnalysis(name=model_pack, providers=["CPUExecutionProvider"])
        app.prepare(ctx_id=-1, det_size=det_size)
        print(f"Model aktif: {model_pack} | provider: CPUExecutionProvider | det_size: {det_size}")
        return app

app = load_insightface_app()

## 4. Utility image, drawing, dan recognition

Pipeline recognition:

1. Deteksi wajah pada frame.
2. Ambil `normed_embedding` dari setiap wajah.
3. Hitung cosine similarity terhadap embedding centroid di facebank.
4. Jika skor terbaik `< SIM_THRESHOLD`, label menjadi `UNKNOWN`.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_image_files(folder):
    folder = Path(folder)
    files = []
    for ext in IMG_EXTS:
        files.extend(folder.rglob(f"*{ext}"))
        files.extend(folder.rglob(f"*{ext.upper()}"))
    return sorted(set(files))

def l2_normalize(x, axis=-1, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=axis, keepdims=True), eps)

def largest_face(faces):
    if len(faces) == 0:
        return None
    return max(
        faces,
        key=lambda f: max(0, f.bbox[2] - f.bbox[0]) * max(0, f.bbox[3] - f.bbox[1])
    )

def read_image_bgr(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Gagal membaca gambar: {path}")
    return img

def recognize_embedding(embedding, bank_names, bank_embeddings, threshold=SIM_THRESHOLD):
    if bank_embeddings is None or len(bank_embeddings) == 0:
        return "NO_FACEBANK", 0.0, -1

    emb = l2_normalize(np.asarray(embedding, dtype=np.float32).reshape(1, -1))[0]
    sims = np.dot(bank_embeddings, emb)
    idx = int(np.argmax(sims))
    score = float(sims[idx])
    name = str(bank_names[idx]) if score >= threshold else "UNKNOWN"
    return name, score, idx

def draw_label(img, x1, y1, text, color, font_scale=0.65, thickness=2):
    # background label agar teks terbaca
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    y_text = max(y1 - 8, th + 8)
    cv2.rectangle(img, (x1, y_text - th - baseline - 6), (x1 + tw + 8, y_text + baseline), color, -1)
    cv2.putText(img, text, (x1 + 4, y_text - 4), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

def recognize_frame(frame_bgr, bank_names=None, bank_embeddings=None, threshold=SIM_THRESHOLD, draw_landmarks=True):
    annotated = frame_bgr.copy()
    faces = app.get(frame_bgr)
    results = []

    for face in faces:
        x1, y1, x2, y2 = face.bbox.astype(int)
        h, w = annotated.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)

        name, score, idx = recognize_embedding(face.normed_embedding, bank_names, bank_embeddings, threshold)
        color = (0, 180, 0) if name not in ["UNKNOWN", "NO_FACEBANK"] else (0, 140, 255)

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        label = f"{name} | {score:.2f}" if name != "NO_FACEBANK" else "NO_FACEBANK"
        draw_label(annotated, x1, y1, label, color)

        if draw_landmarks and hasattr(face, "kps") and face.kps is not None:
            for px, py in face.kps.astype(int):
                cv2.circle(annotated, (int(px), int(py)), 2, (255, 255, 255), -1)

        results.append({
            "name": name,
            "score": score,
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "face_index": idx
        })

    return annotated, results

## 5. Opsi A — siapkan dataset dari Google Drive

Masukkan foto wajah ke `DATASET_ROOT`, satu folder per orang. Contoh:

```text
/content/drive/MyDrive/Colab_Face_Recognition_InsightFace/dataset/
├── nama_orang_1/
    ├── 001.jpg
    ├── 002.jpg
├── nama_orang_2/
    ├── 001.jpg
    ├── 002.jpg
```

Jalankan cell berikut untuk melihat ringkasan dataset.

In [ ]:
def dataset_summary(dataset_root=DATASET_ROOT):
    dataset_root = Path(dataset_root)
    rows = []
    for person_dir in sorted([p for p in dataset_root.iterdir() if p.is_dir()]):
        n = len(list_image_files(person_dir))
        rows.append({"identity": person_dir.name, "num_images": n, "folder": str(person_dir)})
    df = pd.DataFrame(rows)
    if len(df) == 0:
        print("Dataset masih kosong. Tambahkan folder per identitas atau gunakan Opsi B untuk capture dari kamera.")
    else:
        display(df)
    return df

df_dataset = dataset_summary()

## 6. Opsi B — enrollment cepat dari kamera Colab

Gunakan fungsi ini untuk mengambil beberapa sampel wajah dari kamera dan menyimpannya ke folder dataset.  
Pastikan hanya **satu orang** berada di depan kamera saat enrollment.

In [ ]:
COLAB_CAMERA_JS = """
async function startFaceRecCamera(width=640) {
  if (window.faceRecState && window.faceRecState.stream) {
    stopFaceRecCamera();
  }

  const div = document.createElement('div');
  div.setAttribute('style', 'border: 1px solid #ddd; padding: 8px; width: fit-content;');

  const info = document.createElement('div');
  info.innerHTML = '<b>Live camera Colab</b><br>Klik tombol Stop untuk menghentikan kamera.';
  info.setAttribute('style', 'font-family: sans-serif; margin-bottom: 6px;');
  div.appendChild(info);

  const stopButton = document.createElement('button');
  stopButton.textContent = 'Stop camera';
  stopButton.setAttribute('style', 'margin-bottom: 8px;');
  div.appendChild(stopButton);

  const video = document.createElement('video');
  video.setAttribute('playsinline', '');
  video.style.display = 'block';
  video.width = width;
  div.appendChild(video);

  const canvas = document.createElement('canvas');
  canvas.style.display = 'none';
  div.appendChild(canvas);

  const label = document.createElement('div');
  label.textContent = 'Menyiapkan kamera...';
  label.setAttribute('style', 'font-family: monospace; margin: 6px 0; white-space: pre-wrap;');
  div.appendChild(label);

  const output = document.createElement('img');
  output.width = width;
  output.style.display = 'block';
  output.style.border = '1px solid #ddd';
  div.appendChild(output);

  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({
    video: { facingMode: 'user', width: {ideal: width} },
    audio: false
  });

  video.srcObject = stream;
  await video.play();

  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

  window.faceRecState = {
    div: div,
    video: video,
    canvas: canvas,
    output: output,
    label: label,
    stream: stream,
    stopped: false
  };

  stopButton.onclick = () => {
    stopFaceRecCamera();
  };

  label.textContent = 'Kamera aktif.';
  return true;
}

function captureFaceRecFrame(quality=0.80) {
  if (!window.faceRecState || window.faceRecState.stopped) {
    return null;
  }
  const video = window.faceRecState.video;
  const canvas = window.faceRecState.canvas;

  if (video.videoWidth === 0 || video.videoHeight === 0) {
    return null;
  }

  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  const ctx = canvas.getContext('2d');
  ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
  return canvas.toDataURL('image/jpeg', quality);
}

function showFaceRecResult(dataUrl, text) {
  if (!window.faceRecState || window.faceRecState.stopped) {
    return false;
  }
  window.faceRecState.output.src = dataUrl;
  window.faceRecState.label.textContent = text;
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  return true;
}

function stopFaceRecCamera() {
  if (!window.faceRecState) {
    return false;
  }
  const state = window.faceRecState;
  state.stopped = true;
  if (state.stream) {
    state.stream.getTracks().forEach(track => track.stop());
  }
  if (state.label) {
    state.label.textContent = 'Kamera dihentikan.';
  }
  window.faceRecState = null;
  return true;
}
"""

def js_to_bgr(data_url):
    if data_url is None:
        return None
    header, encoded = data_url.split(",", 1)
    img_bytes = base64.b64decode(encoded)
    img_np = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
    return frame

def bgr_to_data_url(img_bgr, quality=85):
    ok, buffer = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        raise ValueError("Gagal encode frame ke JPEG.")
    encoded = base64.b64encode(buffer).decode("utf-8")
    return "data:image/jpeg;base64," + encoded

def capture_enrollment_samples(person_name, num_samples=10, delay=0.8, dataset_root=DATASET_ROOT, width=640):
    """
    Mengambil num_samples foto dari webcam Colab untuk satu identitas.
    Simpan frame mentah; saat build facebank, wajah terbesar akan dideteksi otomatis.
    """
    person_name = str(person_name).strip().replace("/", "_").replace("\\", "_")
    if not person_name:
        raise ValueError("person_name tidak boleh kosong.")

    person_dir = Path(dataset_root) / person_name
    person_dir.mkdir(parents=True, exist_ok=True)

    display(Javascript(COLAB_CAMERA_JS))
    eval_js(f"startFaceRecCamera({int(width)})")
    time.sleep(1.5)

    saved = []
    try:
        for i in range(num_samples):
            time.sleep(delay)
            data_url = eval_js("captureFaceRecFrame(0.90)")
            frame = js_to_bgr(data_url)
            if frame is None:
                print("Frame kosong; ulangi capture.")
                continue

            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            out_path = person_dir / f"{person_name}_{timestamp}_{i+1:03d}.jpg"
            cv2.imwrite(str(out_path), frame)
            saved.append(out_path)

            preview = frame.copy()
            cv2.putText(preview, f"Saved {i+1}/{num_samples}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2, cv2.LINE_AA)
            eval_js(f"showFaceRecResult({json.dumps(bgr_to_data_url(preview))}, {json.dumps(str(out_path))})")

        print(f"Selesai. Tersimpan {len(saved)} sampel di: {person_dir}")
    finally:
        eval_js("stopFaceRecCamera()")

    return saved

# Contoh pakai:
# capture_enrollment_samples("nama_mahasiswa", num_samples=12)

## 7. Build facebank

Facebank adalah database embedding setiap identitas.  
Untuk setiap orang, notebook mengambil embedding dari semua foto yang valid, lalu membuat **centroid embedding** sebagai representasi orang tersebut.

In [ ]:
def build_facebank(dataset_root=DATASET_ROOT, save_path=FACEBANK_PATH, min_faces_per_identity=1):
    dataset_root = Path(dataset_root)
    identities = sorted([p for p in dataset_root.iterdir() if p.is_dir()])

    if len(identities) == 0:
        raise ValueError(f"Tidak ada folder identitas di {dataset_root}. Tambahkan dataset terlebih dahulu.")

    bank_names = []
    bank_embeddings = []
    sample_names = []
    sample_paths = []
    sample_embeddings = []
    rejected = []

    for person_dir in identities:
        person_name = person_dir.name
        image_files = list_image_files(person_dir)

        embs = []
        print(f"\nMemproses {person_name}: {len(image_files)} gambar")
        for img_path in tqdm(image_files, leave=False):
            try:
                img = read_image_bgr(img_path)
                faces = app.get(img)

                if len(faces) == 0:
                    rejected.append({"path": str(img_path), "reason": "no_face"})
                    continue

                face = largest_face(faces)
                emb = np.asarray(face.normed_embedding, dtype=np.float32)
                emb = l2_normalize(emb.reshape(1, -1))[0]
                embs.append(emb)

                sample_names.append(person_name)
                sample_paths.append(str(img_path))
                sample_embeddings.append(emb)

            except Exception as e:
                rejected.append({"path": str(img_path), "reason": repr(e)})

        if len(embs) >= min_faces_per_identity:
            centroid = l2_normalize(np.mean(np.vstack(embs), axis=0, keepdims=True))[0]
            bank_names.append(person_name)
            bank_embeddings.append(centroid)
            print(f"  OK: {len(embs)} embedding valid.")
        else:
            print(f"  SKIP: embedding valid kurang dari {min_faces_per_identity}.")

    if len(bank_embeddings) == 0:
        raise ValueError("Tidak ada embedding valid. Periksa dataset dan kualitas foto.")

    bank_names = np.array(bank_names)
    bank_embeddings = np.vstack(bank_embeddings).astype(np.float32)

    sample_names = np.array(sample_names)
    sample_paths = np.array(sample_paths)
    sample_embeddings = np.vstack(sample_embeddings).astype(np.float32) if len(sample_embeddings) else np.empty((0, 512), dtype=np.float32)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        save_path,
        names=bank_names,
        embeddings=bank_embeddings,
        sample_names=sample_names,
        sample_paths=sample_paths,
        sample_embeddings=sample_embeddings,
        model_pack=np.array([MODEL_PACK]),
        det_size=np.array(DET_SIZE),
        sim_threshold=np.array([SIM_THRESHOLD]),
        created_at=np.array([datetime.now().isoformat()])
    )

    if len(rejected) > 0:
        rejected_path = LOG_DIR / "facebank_rejected_images.csv"
        pd.DataFrame(rejected).to_csv(rejected_path, index=False)
        print(f"Catatan gambar ditolak disimpan di: {rejected_path}")

    print(f"\nFacebank tersimpan: {save_path}")
    print(f"Jumlah identitas: {len(bank_names)}")
    return bank_names, bank_embeddings, sample_names, sample_embeddings, sample_paths

def load_facebank(path=FACEBANK_PATH):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Facebank belum ada: {path}. Jalankan build_facebank() terlebih dahulu.")
    data = np.load(path, allow_pickle=True)
    names = data["names"]
    embeddings = data["embeddings"].astype(np.float32)
    print(f"Facebank loaded: {path}")
    print("Identitas:", list(names))
    return names, embeddings, data

# Jalankan setelah dataset siap:
bank_names, bank_embeddings, sample_names, sample_embeddings, sample_paths = build_facebank()

## 8. Kalibrasi threshold similarity

Ambang default `SIM_THRESHOLD = 0.42` adalah titik awal. Dataset, kamera, dan pencahayaan dapat membuat threshold ideal berbeda.

Gunakan cell ini untuk melihat distribusi similarity:
- **positive pair** = foto orang yang sama
- **negative pair** = foto orang berbeda

Pilih threshold yang memisahkan keduanya sebaik mungkin.

In [ ]:
def similarity_threshold_report(sample_names, sample_embeddings, max_pairs=20000):
    sample_names = np.asarray(sample_names)
    embs = l2_normalize(np.asarray(sample_embeddings, dtype=np.float32))

    if len(embs) < 2:
        print("Sampel terlalu sedikit untuk kalibrasi threshold.")
        return None, None

    pos, neg = [], []
    rng = np.random.default_rng(42)
    n = len(embs)

    # sampling pair agar tidak terlalu berat untuk dataset besar
    pair_count = 0
    attempts = 0
    max_attempts = max_pairs * 5

    while pair_count < max_pairs and attempts < max_attempts:
        i, j = rng.integers(0, n, size=2)
        attempts += 1
        if i == j:
            continue
        sim = float(np.dot(embs[i], embs[j]))
        if sample_names[i] == sample_names[j]:
            pos.append(sim)
        else:
            neg.append(sim)
        pair_count += 1

    print(f"Positive pairs: {len(pos)} | Negative pairs: {len(neg)}")
    if len(pos):
        print("Positive similarity: min/mean/p05/p50 =", np.min(pos), np.mean(pos), np.percentile(pos, 5), np.percentile(pos, 50))
    if len(neg):
        print("Negative similarity: max/mean/p95/p99 =", np.max(neg), np.mean(neg), np.percentile(neg, 95), np.percentile(neg, 99))

    if len(pos) and len(neg):
        suggested = (np.percentile(pos, 5) + np.percentile(neg, 95)) / 2
        print(f"Threshold saran awal: {suggested:.3f}")
        print(f"Threshold aktif saat ini: {SIM_THRESHOLD:.3f}")

    plt.figure(figsize=(8, 4))
    if len(pos):
        plt.hist(pos, bins=30, alpha=0.6, label="same person")
    if len(neg):
        plt.hist(neg, bins=30, alpha=0.6, label="different person")
    plt.axvline(SIM_THRESHOLD, linestyle="--", label=f"threshold={SIM_THRESHOLD}")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Frequency")
    plt.title("Distribusi similarity untuk kalibrasi threshold")
    plt.legend()
    plt.show()

    return pos, neg

# Jalankan setelah facebank dibuat:
pos_scores, neg_scores = similarity_threshold_report(sample_names, sample_embeddings)

## 9. Inference pada gambar statis

Isi `TEST_IMAGE_PATH` dengan path gambar yang ingin diuji.

In [ ]:
# Ubah path ini sesuai gambar uji.
TEST_IMAGE_PATH = "/content/drive/MyDrive/Colab_Face_Recognition_InsightFace/dataset/Jun/1.jpg"

bank_names, bank_embeddings, facebank_data = load_facebank()

if TEST_IMAGE_PATH:
    img = read_image_bgr(TEST_IMAGE_PATH)
    annotated, results = recognize_frame(img, bank_names, bank_embeddings, threshold=SIM_THRESHOLD)
    print(results)
    cv2_imshow(annotated)
else:
    print("Isi TEST_IMAGE_PATH terlebih dahulu jika ingin uji gambar statis.")

## 10. Inference kamera live di Google Colab

Notebook ini memakai JavaScript browser untuk mengambil frame kamera, lalu mengirim frame ke Python untuk diproses.

Parameter:
- `max_frames`: jumlah frame yang diproses sebelum berhenti otomatis.
- `delay`: jeda antar-frame. Naikkan jika koneksi lambat.
- `width`: lebar tampilan video.

In [ ]:
def summarize_results(results):
    if len(results) == 0:
        return "Tidak ada wajah terdeteksi."
    lines = []
    for r in results:
        lines.append(f"{r['name']} | score={r['score']:.3f} | bbox={r['bbox']}")
    return "\n".join(lines)

def run_live_recognition(max_frames=300, delay=0.05, width=640, save_log=True):
    bank_names, bank_embeddings, _ = load_facebank()

    display(Javascript(COLAB_CAMERA_JS))
    eval_js(f"startFaceRecCamera({int(width)})")
    time.sleep(1.5)

    rows = []
    fps_times = []

    try:
        for frame_idx in range(max_frames):
            t0 = time.time()
            data_url = eval_js("captureFaceRecFrame(0.80)")
            frame = js_to_bgr(data_url)

            if frame is None:
                print("Frame kosong atau kamera dihentikan.")
                break

            annotated, results = recognize_frame(
                frame,
                bank_names=bank_names,
                bank_embeddings=bank_embeddings,
                threshold=SIM_THRESHOLD,
                draw_landmarks=True
            )

            elapsed = time.time() - t0
            fps = 1.0 / max(elapsed, 1e-6)
            fps_times.append(fps)

            cv2.putText(
                annotated,
                f"FPS: {fps:.1f} | frame {frame_idx+1}/{max_frames}",
                (15, annotated.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            text = summarize_results(results)
            eval_js(f"showFaceRecResult({json.dumps(bgr_to_data_url(annotated))}, {json.dumps(text)})")

            now = datetime.now().isoformat()
            for r in results:
                rows.append({
                    "time": now,
                    "frame": frame_idx + 1,
                    "name": r["name"],
                    "score": r["score"],
                    "bbox": r["bbox"]
                })

            if delay > 0:
                time.sleep(delay)

        if len(fps_times):
            print(f"FPS rata-rata: {np.mean(fps_times):.2f}")

    finally:
        eval_js("stopFaceRecCamera()")

    if save_log and len(rows) > 0:
        log_path = LOG_DIR / f"live_recognition_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        pd.DataFrame(rows).to_csv(log_path, index=False)
        print(f"Log inference tersimpan: {log_path}")

    return pd.DataFrame(rows)

# Jalankan untuk live recognition:
live_log = run_live_recognition(max_frames=300, delay=0.05, width=640)